In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [2]:
df = pd.read_csv("classification_data.csv")

numeric_cols = [c for c in df.columns if c.startswith("num_")]
cat_cols = [c for c in df.columns if c.startswith("cat_")]

X = df.drop(columns=["target"])
y = df["target"]


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", SelectKBest(score_func=mutual_info_classif, k=15)),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


In [5]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1 score:  {f1:.3f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy:  0.905
Precision: 0.514
Recall:    0.910
F1 score:  0.657
Confusion matrix:
 [[814  86]
 [  9  91]]


In [6]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="f1")
print("CV F1:", cv_scores.mean())


CV F1: 0.6554323591712636


In [7]:
param_grid = {
    "selector__k": [10, 15, 20],
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1")
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)


Best params: {'model__C': 10, 'selector__k': 20}
Best CV F1: 0.7674446962450895


In [8]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Baseline -> F1: {:.3f}, Accuracy: {:.3f}".format(f1, acc))
print("Tuned    -> F1: {:.3f}, Accuracy: {:.3f}".format(
    f1_score(y_test, y_pred_best),
    accuracy_score(y_test, y_pred_best)
))


Baseline -> F1: 0.657, Accuracy: 0.905
Tuned    -> F1: 0.720, Accuracy: 0.926
